In [2]:
pip install shap

  Using cached shap-0.51.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (25 kB)
  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached pandas-3.0.2-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached numba-0.65.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (2.9 kB)
  Using cached llvmlite-0.47.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached shap-0.51.0-cp311-cp311-macosx_11_0_arm64.whl (562 kB)
Using cached slicer-0.0.8-py3-none-any.whl (15 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached llvmlite-0.47.0-cp311-cp311-macosx_11_0_arm64.whl (37.2 MB)
Using

In [4]:
pip install matplotlib

  Using cached matplotlib-3.10.8-cp311-cp311-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp311-cp311-macosx_10_9_universal2.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.1 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached matplotlib-3.10.8-cp311-cp311-macosx_11_0_arm64.whl (8.1 MB)
Using cached contourpy-1.3.3-cp311-cp311-macosx_11_0_arm64.whl (270 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.62.1-cp311-cp311-macosx_10_9_universal2.whl (2.9 MB)
Using cached kiwisolver-1.5.0-cp311-cp311-macosx_11_0_arm64.whl (63 kB)
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib] 5/6 [matplotlib]
Note: you may need to restart the kernel to use 

In [3]:
"""
XAI Multimodal SHAP Analysis
Guava Maturity Classification — RGB + Thermal Late Fusion
==========================================================
Fixes applied vs previous version:
  1. ResNet-50 fc head: auto-detects Sequential vs plain Linear from checkpoint keys
  2. DenseNet-121 classifier: same auto-detection
  3. DenseNetFeatureExtractor: adds ReLU + AdaptiveAvgPool2d so features are (N,1024)
  4. fusion_predict: uses actual classifier weights for real logits → probabilities
  5. SHAP summary_plot: shap_values passed as LIST of per-class arrays (required by
     shap.summary_plot when multi-output). Averaged summary uses a single 2-D array.
  6. All plt figures explicitly closed to avoid memory leaks.
"""

import os
import warnings
import numpy as np
import torch
import torch.nn.functional as F
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────
# DEVICE
# ──────────────────────────────────────────────
DEVICE = torch.device("cpu")

# ──────────────────────────────────────────────
# CONFIG  — edit paths here only
# ──────────────────────────────────────────────
IMG_SIZE    = 224
NUM_CLASSES = 3
CLASS_NAMES = ["mature", "semi_Mature", "Immature"]

DIGITAL_TEST_DIR = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project"
    r"/Dataset/Maturity/train test val split for digital/test"
)
THERMAL_TEST_DIR = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project"
    r"/Dataset/Maturity/train test val split  thermal/test"
)
RGB_CKPT = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project"
    r"/DeepLearningModels/digital_best_resnet50_guava.pth"
)
THERMAL_CKPT = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project"
    r"/DeepLearningModels/densenet121_thermal_best.pth"
)
OUTPUT_DIR = (
    r"/Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multimodal_2"
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_BACKGROUND = 50   # SHAP background samples
MAX_EXPLAIN    = 100  # samples to explain (keep low for speed)

# ──────────────────────────────────────────────
# TRANSFORM
# ──────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# ──────────────────────────────────────────────
# UTILITIES
# ──────────────────────────────────────────────

def _find_linear(module: torch.nn.Module):
    """Return the last nn.Linear found (depth-first) inside module."""
    last = None
    for m in module.modules():
        if isinstance(m, torch.nn.Linear):
            last = m
    return last


def _build_head(in_features: int, num_classes: int, ckpt: dict, key_prefix: str):
    """
    Inspect checkpoint keys to decide whether the saved head was:
      (a) plain Linear                    -> keys: <prefix>.weight / <prefix>.bias
      (b) Sequential(Dropout, Linear)     -> keys: <prefix>.1.weight / <prefix>.1.bias
    Returns a new nn.Module that matches the checkpoint.
    """
    seq_key   = f"{key_prefix}.1.weight"
    plain_key = f"{key_prefix}.weight"

    if seq_key in ckpt:
        return torch.nn.Sequential(
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(in_features, num_classes),
        )
    elif plain_key in ckpt:
        return torch.nn.Linear(in_features, num_classes)
    else:
        # Last resort: scan for any key ending in .weight under this prefix
        candidates = [k for k in ckpt if k.startswith(key_prefix) and k.endswith(".weight")]
        if candidates:
            deepest = max(candidates, key=lambda k: k.count("."))
            w = ckpt[deepest]
            return torch.nn.Linear(in_features, w.shape[0])
        raise KeyError(
            f"Cannot find classifier weights for prefix '{key_prefix}' in checkpoint. "
            f"Available keys (first 20): {list(ckpt.keys())[:20]}"
        )


def load_resnet50(path: str, num_classes: int = NUM_CLASSES) -> torch.nn.Module:
    ckpt  = torch.load(path, map_location=DEVICE)
    model = models.resnet50(pretrained=False)
    model.fc = _build_head(model.fc.in_features, num_classes, ckpt, "fc")
    model.load_state_dict(ckpt, strict=True)
    return model.to(DEVICE).eval()


def load_densenet121(path: str, num_classes: int = NUM_CLASSES) -> torch.nn.Module:
    ckpt  = torch.load(path, map_location=DEVICE)
    model = models.densenet121(pretrained=False)
    model.classifier = _build_head(
        model.classifier.in_features, num_classes, ckpt, "classifier"
    )
    model.load_state_dict(ckpt, strict=True)
    return model.to(DEVICE).eval()


# ──────────────────────────────────────────────
# FEATURE EXTRACTORS
# ──────────────────────────────────────────────

class ResNetFeatureExtractor(torch.nn.Module):
    """ResNet-50 up to and including avgpool -> (N, 2048)."""
    def __init__(self, model):
        super().__init__()
        # children: conv1,bn1,relu,maxpool, layer1..4, avgpool, fc
        self.backbone = torch.nn.Sequential(*list(model.children())[:-1])  # drop fc

    def forward(self, x):
        x = self.backbone(x)
        return torch.flatten(x, 1)


class DenseNetFeatureExtractor(torch.nn.Module):
    """DenseNet-121 features + relu + adaptive pool -> (N, 1024)."""
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.pool     = torch.nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.features(x)
        x = F.relu(x, inplace=True)
        x = self.pool(x)
        return torch.flatten(x, 1)


# ──────────────────────────────────────────────
# LOAD MODELS
# ──────────────────────────────────────────────
print("Loading RGB model  (ResNet-50)...")
rgb_model = load_resnet50(RGB_CKPT)

print("Loading Thermal model (DenseNet-121)...")
thermal_model = load_densenet121(THERMAL_CKPT)

rgb_extractor     = ResNetFeatureExtractor(rgb_model)
thermal_extractor = DenseNetFeatureExtractor(thermal_model)

# Verify feature dimensions with a dummy forward pass
with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
    NUM_RGB_FEATS     = rgb_extractor(_dummy).shape[1]
    NUM_THERMAL_FEATS = thermal_extractor(_dummy).shape[1]
    NUM_TOTAL_FEATS   = NUM_RGB_FEATS + NUM_THERMAL_FEATS

print(f"   RGB feature dim     : {NUM_RGB_FEATS}")
print(f"   Thermal feature dim : {NUM_THERMAL_FEATS}")
print(f"   Total fusion dim    : {NUM_TOTAL_FEATS}")

# ──────────────────────────────────────────────
# CLASSIFIER WEIGHTS  (for fusion_predict)
# ──────────────────────────────────────────────

def get_last_linear_weights(module):
    lin = _find_linear(module)
    if lin is None:
        raise RuntimeError("No Linear layer found in module.")
    return lin.weight.detach().cpu(), lin.bias.detach().cpu()

rgb_W, rgb_b         = get_last_linear_weights(rgb_model.fc)
thermal_W, thermal_b = get_last_linear_weights(thermal_model.classifier)

# Verify classifier output matches NUM_CLASSES
assert rgb_W.shape[0]     == NUM_CLASSES, f"RGB classifier output {rgb_W.shape[0]} != {NUM_CLASSES}"
assert thermal_W.shape[0] == NUM_CLASSES, f"Thermal classifier output {thermal_W.shape[0]} != {NUM_CLASSES}"

# ──────────────────────────────────────────────
# IMAGE LOADER
# ──────────────────────────────────────────────

def load_tensor(path: str) -> torch.Tensor:
    return transform(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)


# ──────────────────────────────────────────────
# EXTRACT FEATURES
# ──────────────────────────────────────────────
print("\nExtracting features from test set...")

rgb_feats_list, thermal_feats_list, label_list = [], [], []

for cls_idx, cls in enumerate(CLASS_NAMES):
    rgb_dir     = os.path.join(DIGITAL_TEST_DIR, cls)
    thermal_dir = os.path.join(THERMAL_TEST_DIR, cls)

    if not os.path.isdir(rgb_dir):
        print(f"   WARNING: Missing RGB dir: {rgb_dir}")
        continue
    if not os.path.isdir(thermal_dir):
        print(f"   WARNING: Missing thermal dir: {thermal_dir}")
        continue

    common = sorted(
        set(os.listdir(rgb_dir)).intersection(os.listdir(thermal_dir))
    )
    if not common:
        print(f"   WARNING: No paired images for class '{cls}'")
        continue

    for fname in tqdm(common, desc=f"   {cls}"):
        try:
            rgb_t = load_tensor(os.path.join(rgb_dir,     fname))
            thm_t = load_tensor(os.path.join(thermal_dir, fname))
        except Exception as exc:
            print(f"      Skipping {fname}: {exc}")
            continue

        with torch.no_grad():
            rf = rgb_extractor(rgb_t).cpu().numpy()[0]
            tf = thermal_extractor(thm_t).cpu().numpy()[0]

        assert rf.shape[0] == NUM_RGB_FEATS
        assert tf.shape[0] == NUM_THERMAL_FEATS

        rgb_feats_list.append(rf)
        thermal_feats_list.append(tf)
        label_list.append(cls_idx)

if not rgb_feats_list:
    raise RuntimeError("No features extracted — please check dataset paths.")

rgb_feats     = np.array(rgb_feats_list,     dtype=np.float32)
thermal_feats = np.array(thermal_feats_list, dtype=np.float32)
labels        = np.array(label_list,         dtype=np.int32)

X_fusion = np.concatenate([rgb_feats, thermal_feats], axis=1)

assert X_fusion.shape[1] == NUM_TOTAL_FEATS, \
    f"X_fusion width {X_fusion.shape[1]} != {NUM_TOTAL_FEATS}"

print(f"\n   Samples collected  : {len(labels)}")
print(f"   X_fusion shape     : {X_fusion.shape}")

# ──────────────────────────────────────────────
# FUSION PREDICT  ->  (N, NUM_CLASSES)  float32
# ──────────────────────────────────────────────

def fusion_predict(X: np.ndarray) -> np.ndarray:
    """
    Late-fusion: average softmax probabilities from both modalities.
    Input  : numpy float32  (N, NUM_RGB_FEATS + NUM_THERMAL_FEATS)
    Output : numpy float32  (N, NUM_CLASSES)
    """
    Xr = torch.tensor(X[:, :NUM_RGB_FEATS],  dtype=torch.float32)
    Xt = torch.tensor(X[:, NUM_RGB_FEATS:],  dtype=torch.float32)

    with torch.no_grad():
        rgb_probs     = torch.softmax(Xr @ rgb_W.T     + rgb_b,     dim=1)
        thermal_probs = torch.softmax(Xt @ thermal_W.T + thermal_b, dim=1)
        fused = ((rgb_probs + thermal_probs) / 2.0).numpy()

    return fused.astype(np.float32)   # (N, NUM_CLASSES)


# Sanity checks
_test_out = fusion_predict(X_fusion[:3])
assert _test_out.shape == (3, NUM_CLASSES), \
    f"fusion_predict shape mismatch: {_test_out.shape}"
assert _test_out.dtype == np.float32
print(f"\nfusion_predict OK -> output shape {_test_out.shape}")

# ──────────────────────────────────────────────
# SHAP
# ──────────────────────────────────────────────
n_explain = min(MAX_EXPLAIN, len(X_fusion))
X_explain = X_fusion[:n_explain].astype(np.float32)

n_bg = min(MAX_BACKGROUND, len(X_fusion))
bg_idx     = np.random.choice(len(X_fusion), n_bg, replace=False)
background = X_fusion[bg_idx].astype(np.float32)

print(f"\nRunning SHAP  (background={n_bg}, explain={n_explain})...")

explainer   = shap.KernelExplainer(fusion_predict, background)

# shap_values is a LIST of NUM_CLASSES arrays, each shape (n_explain, n_features)
shap_values = explainer.shap_values(X_explain)

# --- normalise output: KernelExplainer with multi-output returns a list ---
# Guarantee it's a list of 2-D numpy arrays
if isinstance(shap_values, np.ndarray):
    # shape could be (N,F) single-output or (N,F,C) stacked
    if shap_values.ndim == 3:
        shap_values = [shap_values[:, :, c] for c in range(shap_values.shape[2])]
    else:
        shap_values = [shap_values]

assert len(shap_values) == NUM_CLASSES, \
    f"Expected {NUM_CLASSES} SHAP arrays, got {len(shap_values)}"
for i, sv in enumerate(shap_values):
    assert sv.shape == X_explain.shape, \
        (f"Class {i} SHAP shape {sv.shape} != X_explain shape {X_explain.shape}. "
         "This would crash summary_plot — aborting early.")

print(f"SHAP complete - each array shape: {shap_values[0].shape}")

# ──────────────────────────────────────────────
# FEATURE NAMES
# ──────────────────────────────────────────────
feature_names = (
    [f"RGB_{i}"     for i in range(NUM_RGB_FEATS)]     +
    [f"Thermal_{i}" for i in range(NUM_THERMAL_FEATS)]
)
assert len(feature_names) == X_explain.shape[1], \
    f"feature_names length {len(feature_names)} != {X_explain.shape[1]}"

# ──────────────────────────────────────────────
# PLOT HELPER
# ──────────────────────────────────────────────

def save_fig(path: str):
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close("all")
    print(f"   Saved -> {path}")


# ──────────────────────────────────────────────
# PLOT 1 — Averaged summary
# shap.summary_plot requires:
#   single-output : 2-D array  (n_samples, n_features)
#   multi-output  : list of 2-D arrays
# For the averaged view we compute the mean over classes -> 2-D array.
# ──────────────────────────────────────────────
print("\nPlot 1: Summary (averaged over classes)...")

shap_stack = np.stack(shap_values, axis=0)   # (C, N, F)
shap_avg   = shap_stack.mean(axis=0)          # (N, F)  <- 2-D, matches X_explain shape

assert shap_avg.shape == X_explain.shape, \
    f"shap_avg {shap_avg.shape} != X_explain {X_explain.shape}"

plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_avg,           # 2-D (N, F)
    X_explain,          # 2-D (N, F)
    feature_names=feature_names,
    max_display=20,
    show=False,
)
plt.title("SHAP Summary - Averaged Over Classes", fontsize=13, pad=10)
save_fig(os.path.join(OUTPUT_DIR, "shap_summary_averaged.png"))

# ──────────────────────────────────────────────
# PLOT 2 — Per-class dot / beeswarm plots
# ──────────────────────────────────────────────
print("\nPlot 2: Per-class dot plots...")

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    sv = shap_values[cls_idx]              # (N, F)
    assert sv.shape == X_explain.shape

    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        sv,            # 2-D (N, F)
        X_explain,     # 2-D (N, F)
        feature_names=feature_names,
        max_display=20,
        show=False,
    )
    plt.title(f"SHAP Summary - Class: {cls_name}", fontsize=13, pad=10)
    save_fig(os.path.join(OUTPUT_DIR, f"shap_dot_{cls_name}.png"))

# ──────────────────────────────────────────────
# PLOT 3 — Per-class bar plots
# ──────────────────────────────────────────────
print("\nPlot 3: Per-class bar plots...")

for cls_idx, cls_name in enumerate(CLASS_NAMES):
    sv = shap_values[cls_idx]

    plt.figure(figsize=(10, 5))
    shap.summary_plot(
        sv,
        X_explain,
        feature_names=feature_names,
        max_display=20,
        plot_type="bar",
        show=False,
    )
    plt.title(f"SHAP Bar - Class: {cls_name}", fontsize=13, pad=10)
    save_fig(os.path.join(OUTPUT_DIR, f"shap_bar_{cls_name}.png"))

# ──────────────────────────────────────────────
# PLOT 4 — Modality Contribution
# ──────────────────────────────────────────────
print("\nPlot 4: Modality contribution chart...")

mean_abs = np.abs(shap_stack).mean(axis=(0, 1))   # (F,)

rgb_importance     = float(mean_abs[:NUM_RGB_FEATS].sum())
thermal_importance = float(mean_abs[NUM_RGB_FEATS:].sum())
total              = rgb_importance + thermal_importance

print(f"   RGB Contribution     : {rgb_importance:.4f}  "
      f"({100 * rgb_importance / total:.1f}%)")
print(f"   Thermal Contribution : {thermal_importance:.4f}  "
      f"({100 * thermal_importance / total:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
bar_labels = ["RGB\n(ResNet-50)", "Thermal\n(DenseNet-121)"]
bar_values = [rgb_importance, thermal_importance]
bar_colors = ["#4C72B0", "#DD8452"]

bars = ax.bar(bar_labels, bar_values, color=bar_colors,
              edgecolor="white", linewidth=0.8, width=0.45)

for bar, val in zip(bars, bar_values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + total * 0.01,
        f"{val:.4f}\n({100 * val / total:.1f}%)",
        ha="center", va="bottom", fontsize=10,
    )

ax.set_ylabel("Mean |SHAP value|", fontsize=11)
ax.set_title("Modality Contribution to Fusion Prediction", fontsize=12, pad=10)
ax.set_ylim(0, max(bar_values) * 1.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
save_fig(os.path.join(OUTPUT_DIR, "modality_contribution.png"))

# ──────────────────────────────────────────────
# DONE
# ──────────────────────────────────────────────
print(f"\nAll outputs written to:\n   {OUTPUT_DIR}")

Loading RGB model  (ResNet-50)...
Loading Thermal model (DenseNet-121)...
   RGB feature dim     : 2048
   Thermal feature dim : 1024
   Total fusion dim    : 3072

Extracting features from test set...


   Immature: 100%|██████████| 462/462 [00:33<00:00, 13.87it/s]



   Samples collected  : 1287
   X_fusion shape     : (1287, 3072)

fusion_predict OK -> output shape (3, 3)

Running SHAP  (background=50, explain=100)...


100%|██████████| 100/100 [06:18<00:00,  3.78s/it]


SHAP complete - each array shape: (100, 3072)

Plot 1: Summary (averaged over classes)...
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multimodal_2/shap_summary_averaged.png

Plot 2: Per-class dot plots...
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multimodal_2/shap_dot_mature.png
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multimodal_2/shap_dot_semi_Mature.png
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multimodal_2/shap_dot_Immature.png

Plot 3: Per-class bar plots...
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multimodal_2/shap_bar_mature.png
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multimodal_2/shap_bar_semi_Mature.png
   Saved -> /Users/dhruviramaiya/Downloads/Mtech Major Project/XAI_Multimodal_2/shap_bar_Immature.png

Plot 4: Modality contribution chart...
   RGB Contribution     : 0.0424  (16.3%)
   Thermal Contribution : 0.2173  (83.7%)
  